In [1]:
from pathlib import Path
from glob import glob
import pickle
import re

import pandas as pd
import numpy as np

In [2]:
from bikipy import DeepLabCutReader
from bikipy.preferance import NortObject

In [3]:
WORKING_DIR = Path("C:/Users/Can/Projects/Neuroscience/Imen/data")
NORT_DIR = WORKING_DIR / "nort"
BEFORE_DIR = NORT_DIR / "NORT_02.06.2020"
AFTER_DIR = NORT_DIR / "NORT_24 08 2020 (after)_2"

ANNOTATIONS_PATH = str(WORKING_DIR / "python_data" / "annotations" / "all.pickle")
BEFORE_ANNOTATIONS_PATH = str(WORKING_DIR / "python_data" / "annotations" / "before.pickle")
AFTER_ANNOTATIONS_PATH = str(WORKING_DIR / "python_data" / "annotations" / "after.pickle")
IMPORTED_DLC_FILES = str(WORKING_DIR / "python_data" / "dlc.pickle")
RESULTS = str(WORKING_DIR / "python_data" / "results.pickle")

EXP_ID_FINDER = re.compile("\d+")
BORDER_DISTANCE = 3 * 224 / 40

## Before

### T1

In [4]:
# b_1 = NortObject.from_video(str(BEFORE_DIR / "Test 47.mp4"), labels=("A", "B"), border_distance=BORDER_DISTANCE)
# b_2 = NortObject.from_video(str(BEFORE_DIR / "Test 49.mp4"), labels=("B", "A"), border_distance=BORDER_DISTANCE)
# b_3 = NortObject.from_video(str(BEFORE_DIR / "Test 48.mp4"), labels=("B", "A"), border_distance=BORDER_DISTANCE)
# b_4 = NortObject.from_video(str(BEFORE_DIR / "Test 50.mp4"), labels=("A", "B"), frame_time="start", border_distance=BORDER_DISTANCE)

### T2

In [5]:
# b2_1 = NortObject.from_video(str(BEFORE_DIR / "Test 96.mp4"), labels=("A", "B"), border_distance=BORDER_DISTANCE)
# b2_2 = NortObject.from_video(str(BEFORE_DIR / "Test 93.mp4"), labels=("B", "A"), border_distance=BORDER_DISTANCE)
# b2_3 = NortObject.from_video(str(BEFORE_DIR / "Test 95.mp4"), labels=("B", "A"), border_distance=BORDER_DISTANCE)
# b2_4 = NortObject.from_video(str(BEFORE_DIR / "Test 94.mp4"), labels=("A", "B"), border_distance=BORDER_DISTANCE)

## After

### T1

In [6]:
# a1_1 = NortObject.from_video(str(AFTER_DIR / "Test 44.mp4"), labels=("A", "B"), frame_time="start", border_distance=BORDER_DISTANCE)
# a1_2 = NortObject.from_video(str(AFTER_DIR / "Test 46.mp4"), labels=("B", "A"), border_distance=BORDER_DISTANCE)
# a1_3 = NortObject.from_video(str(AFTER_DIR / "Test 45.mp4"), labels=("B", "A"), border_distance=BORDER_DISTANCE)
# a1_4 = NortObject.from_video(str(AFTER_DIR / "Test 47.mp4"), labels=("A", "B"), border_distance=BORDER_DISTANCE)

# with open(AFTER_ANNOTATIONS_PATH, "wb") as outfile:
#     pickle.dump((a_1, a_2, a_3, a_4), outfile)

### T2

In [7]:
# a2_1 = NortObject.from_video(str(AFTER_DIR / "Test 90.mp4"), labels=("A", "B"), border_distance=BORDER_DISTANCE)
# a2_2 = NortObject.from_video(str(AFTER_DIR / "Test 89.mp4"), labels=("B", "A"), border_distance=BORDER_DISTANCE)
# a2_3 = NortObject.from_video(str(AFTER_DIR / "Test 87.mp4"), labels=("B", "A"), border_distance=BORDER_DISTANCE)
# a2_4 = NortObject.from_video(str(AFTER_DIR / "Test 88.mp4"), labels=("A", "B"), border_distance=BORDER_DISTANCE)

In [8]:
# with open(AFTER_ANNOTATIONS_PATH, "wb") as outfile:
#     pickle.dump((a1_1, a1_2, a1_3, a1_4, a2_1, a2_2, a2_3, a2_4), outfile)

In [9]:
#with open(ANNOTATIONS_PATH, "wb") as outfile:
#    pickle.dump((a1_1, a1_2, a1_3, a1_4, a2_1, a2_2, a2_3, a2_4, b1_1, b1_2, b1_3, b1_4, b2_1, b2_2, b2_3, b2_4), outfile)

In [10]:
with open(ANNOTATIONS_PATH, "rb") as infile:
    a1_1, a1_2, a1_3, a1_4, a2_1, a2_2, a2_3, a2_4, b1_1, b1_2, b1_3, b1_4, b2_1, b2_2, b2_3, b2_4 = pickle.load(infile)

In [11]:
app_to_obj_0_1 = {1: b1_1, 2: b1_2, 3: b1_3, 4: b1_4} 
app_to_obj_1_1 = {1: a1_1, 2: a1_2, 3: a1_3, 4: a1_4}
app_to_obj_0_2 = {1: b2_1, 2: b2_2, 3: b2_3, 4: b2_4} 
app_to_obj_1_2 = {1: a2_1, 2: a2_2, 3: a2_3, 4: a2_4} 

In [12]:
exp_info_df_0 = pd.read_excel(str(WORKING_DIR / "NORT_Round1.xlsx"), sheet_name=0)
exp_info_df_1 = pd.read_excel(str(WORKING_DIR / "NORT_Round1.xlsx"), sheet_name=1)

In [13]:
def get_animal_id_vs_exp_ids(info_df):
    exp_ids = np.array([int(EXP_ID_FINDER.findall(info)[1]) for info in info_df["Video_file_name"]])
    animal_id = np.array(info_df["Animal"])
    return {i: tuple(exp_ids[np.where(animal_id == i)[0][:2]]) for i in range(int(animal_id.min()), int(animal_id.max()))}

In [14]:
id_exp_0 = get_animal_id_vs_exp_ids(exp_info_df_0)
id_exp_1 = get_animal_id_vs_exp_ids(exp_info_df_1)

In [15]:
def get_animal_id_vs_apparatus(info_df):
    animal_id = np.unique(info_df["Animal"])
    apparatus = info_df["Apparatus"]
    
    return {i: EXP_ID_FINDER.findall(app)[0] for i, app in zip(animal_id, apparatus)}

In [16]:
id_app_0 = get_animal_id_vs_apparatus(exp_info_df_0)
id_app_1 = get_animal_id_vs_apparatus(exp_info_df_1)

In [17]:
def dlc_objectifier(dir_path):
    exp_id_vs_dlc = {}
    for h_file in glob(str(dir_path / "*.h5")):
        exp_id = int(EXP_ID_FINDER.findall(Path(h_file).stem)[0])
        exp_id_vs_dlc[exp_id] = DeepLabCutReader.from_video(
            str(dir_path / f"Test {exp_id}.mp4"), hdf_path=h_file,
            midpoint_groups=[["left_ear", "right_ear"]]
        )
    return exp_id_vs_dlc

In [18]:
# id_dlc_0 = dlc_objectifier(BEFORE_DIR)
# id_dlc_1 = dlc_objectifier(AFTER_DIR)

In [19]:
# with open(IMPORTED_DLC_FILES, "wb") as outfile:
#     pickle.dump((id_dlc_0, id_dlc_1), outfile)

In [20]:
with open(IMPORTED_DLC_FILES, "rb") as infile:
    id_dlc_0, id_dlc_1 = pickle.load(infile)

In [21]:
def analyse(id_exp, app_to_obj_t1, app_to_obj_t2):
    result = {}
    for animal, exp_ids in id_exp.items():
        app_A_t1 = app_to_obj_t1[animal]["A"]
        app_B_t1 = app_to_obj_t1[animal]["B"]
        app_A_t2 = app_to_obj_t2[animal]["A"]
        app_B_t2 = app_to_obj_t2[animal]["B"]
        t1 = id_dlc_0[exp_ids[0]]
        t2 = id_dlc_0[exp_ids[1]]

        t1_A_at = app_A_t1.attention(t1["mid-left_ear-right_ear"], t1["nose"], 14.99)
        t1_B_at = app_B_t1.attention(t1["mid-left_ear-right_ear"], t1["nose"], 14.99)
        t2_A_at = app_A_t2.attention(t2["mid-left_ear-right_ear"], t2["nose"], 14.99)
        t2_B_at = app_B_t2.attention(t2["mid-left_ear-right_ear"], t2["nose"], 14.99)
        return t1, t2

In [ ]:
results_0 = analyse(id_exp_0, app_to_obj_0_1, app_to_obj_0_2)